# Chapter 12 — Running time (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch12_running_time_python.ipynb)

This notebook goes with chapter 12 «Running time» of the lecture notes (volume 4, *Analysis and recursion*) and with its [page of the site](https://fabiofurini.github.io/algorithms/analysis/12-running-time/). It contains the **Python** code of the algorithms with pseudo-code of the chapter: `FindMaximum`, `InsertionSort`, `ChangeToBase2`, `PrimalityTest`. The running time is measured in **steps**: a step is the execution of one line of pseudo-code, and the condition of a `while` counts as one step each time it is evaluated. The lines are numbered as in the lecture notes (line 1 is the `function` header) and an array `cnt` counts how many times each line is executed: the printed tables are those of the chapter.

**How to run it.** Click the «Open in Colab» badge, then *Runtime ▸ Run all* (or cell by cell with Shift+Enter). Nothing to install.

Every algorithm is a Python function; the run cells execute it on the same examples as the lecture notes and print the results.

**Indices.** In the pseudo-code the arrays are indexed from 1; in the Python code from 0: the element `a[i]` of the pseudo-code is `a[i-1]` in the code (and a loop `while i <= n` from 1 becomes `while i < n` from 0). The printed tables use the indices of the pseudo-code.

## Tools: the step counter

The cell defines the array `cnt` (one entry per line of the pseudo-code), `reset()` to clear it and `print_steps()` to print the table of executions.

In [1]:
cnt = [0] * 20      # cnt[r] = number of executions of line r (lines numbered as in the lecture notes)


def reset():
    for r in range(len(cnt)):
        cnt[r] = 0


def print_steps(*lines):
    print("Line | Executions")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Total steps S = %d" % sum(cnt))

## 1. Find maximum: FindMaximum

**Input:** an array $a[1..n]$ of $n \ge 1$ integer numbers. **Output:** the maximum value of $a$. In the worst case (increasing array) $S(n) = 4n = \Theta(n)$.

Pseudo-code: section 12.2 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/12-running-time/).

In [2]:
def find_maximum(a):
    n = len(a)
    max_val = a[0]; cnt[2] += 1        # max := a[1]
    i = 1; cnt[3] += 1                 # i := 2 in the pseudo-code
    cnt[4] += 1                        # the while condition is counted at every evaluation
    while i < n:                       # i <= n in the pseudo-code
        cnt[5] += 1
        if a[i] > max_val:
            max_val = a[i]; cnt[6] += 1
        i = i + 1; cnt[8] += 1
        cnt[4] += 1
    cnt[10] += 1
    return max_val

In [3]:
def steps_of(a):
    reset()
    find_maximum(a)
    return sum(cnt)


# Example of the lecture notes: a[1..4] = (3, 1, 4, 5)
a = [3, 1, 4, 5]
n = len(a)
reset()
m = find_maximum(a)
print("a = %s -> max = %d" % (a, m))
print_steps(2, 3, 4, 5, 6, 8, 10)
t = cnt[6]
print("sum t(i) = %d, S(n) = 3n + 1 + sum t(i) = %d" % (t, 3 * n + 1 + t))

# Worst case (increasing array) and best case (decreasing array)
print()
a = [1, 2, 3, 4]
print("a = %s (increasing): S = %d = 4n = %d" % (a, steps_of(a), 4 * n))
a = [4, 3, 2, 1]
print("a = %s (decreasing): S = %d = 3n + 1 = %d" % (a, steps_of(a), 3 * n + 1))

print()
print("Increasing array (1, 2, ..., n):")
print("  n | S(n) | 4n")
for n in range(1, 9):
    print("%3d | %4d | %d" % (n, steps_of(list(range(1, n + 1))), 4 * n))

a = [3, 1, 4, 5] -> max = 5
Line | Executions
   2 |          1
   3 |          1
   4 |          4
   5 |          3
   6 |          2
   8 |          3
  10 |          1
Total steps S = 15
sum t(i) = 2, S(n) = 3n + 1 + sum t(i) = 15

a = [1, 2, 3, 4] (increasing): S = 16 = 4n = 16
a = [4, 3, 2, 1] (decreasing): S = 13 = 3n + 1 = 13

Increasing array (1, 2, ..., n):
  n | S(n) | 4n
  1 |    4 | 4
  2 |    8 | 8
  3 |   12 | 12
  4 |   16 | 16
  5 |   20 | 20
  6 |   24 | 24
  7 |   28 | 28
  8 |   32 | 32


**Try it yourself.** Change the array: the steps are always between $3n + 1$ and $4n$.

In [4]:
a = [7, 2, 9, 4, 9, 1]      # <-- change the array
n = len(a)
reset()
m = find_maximum(a)
print("a = %s -> max = %d" % (a, m))
print("S = %d, between 3n + 1 = %d and 4n = %d" % (sum(cnt), 3 * n + 1, 4 * n))

a = [7, 2, 9, 4, 9, 1] -> max = 9
S = 20, between 3n + 1 = 19 and 4n = 24


## 2. Insertion sort: InsertionSort

**Input:** an array $a[1..n]$ of $n$ natural numbers. **Output:** the array $a$ sorted in non-decreasing order. If $t(i)$ is the number of shifts at iteration $i$, then $S(n) = 6n - 4 + 3\sum_{i=2}^{n} t(i)$; in the worst case (decreasing array) $S(n) = \frac{3n^2+9n-8}{2} = \Theta(n^2)$.

Pseudo-code: section 12.3 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/12-running-time/).

In [5]:
def insertion_sort(a, trace=False):
    n = len(a)
    prev = 0                               # only for the trace
    i = 1; cnt[2] += 1                     # i := 2 in the pseudo-code
    cnt[3] += 1
    while i < n:                           # i <= n in the pseudo-code
        key = a[i]; cnt[4] += 1
        j = i - 1; cnt[5] += 1
        cnt[6] += 1                        # the while condition is counted at every evaluation
        while j >= 0 and a[j] > key:       # j >= 1 and a[j] > key in the pseudo-code
            a[j + 1] = a[j]; cnt[7] += 1
            j = j - 1; cnt[8] += 1
            cnt[6] += 1
        a[j + 1] = key; cnt[10] += 1
        i = i + 1; cnt[11] += 1
        cnt[3] += 1
        if trace:                          # trace: after the increment, i is the pseudo-code index of the iteration just finished
            print("%2d | %3d | %4d | %s" % (i, key, cnt[7] - prev, a))
            prev = cnt[7]

In [6]:
def steps_of(a):
    reset()
    insertion_sort(a)
    return sum(cnt)


# Example of the lecture notes: a[1..6] = (5, 2, 4, 6, 1, 3)
a = [5, 2, 4, 6, 1, 3]
n = len(a)
print("a = %s" % a)
reset()
print(" i | key | t(i) | array after the iteration")
insertion_sort(a, True)
print("sorted: %s" % a)
print_steps(2, 3, 4, 5, 6, 7, 8, 10, 11)
t = cnt[7]
print("sum t(i) = %d, S(n) = 6n - 4 + 3 sum t(i) = %d" % (t, 6 * n - 4 + 3 * t))

# Worst case: decreasing array, t(i) = i - 1
print()
a = [6, 5, 4, 3, 2, 1]
print("a = %s (decreasing): S = %d = (3n^2 + 9n - 8)/2 = %d" % (a, steps_of(a[:]), (3 * n * n + 9 * n - 8) // 2))
print("sum t(i) = %d = n(n-1)/2 = %d" % (cnt[7], n * (n - 1) // 2))

print()
print("Decreasing array (n, ..., 2, 1):")
print("  n | S(n) | (3n^2 + 9n - 8)/2")
for n in range(1, 9):
    print("%3d | %4d | %d" % (n, steps_of(list(range(n, 0, -1))), (3 * n * n + 9 * n - 8) // 2))

a = [5, 2, 4, 6, 1, 3]
 i | key | t(i) | array after the iteration
 2 |   2 |    1 | [2, 5, 4, 6, 1, 3]
 3 |   4 |    1 | [2, 4, 5, 6, 1, 3]
 4 |   6 |    0 | [2, 4, 5, 6, 1, 3]
 5 |   1 |    4 | [1, 2, 4, 5, 6, 3]
 6 |   3 |    3 | [1, 2, 3, 4, 5, 6]
sorted: [1, 2, 3, 4, 5, 6]
Line | Executions
   2 |          1
   3 |          6
   4 |          5
   5 |          5
   6 |         14
   7 |          9
   8 |          9
  10 |          5
  11 |          5
Total steps S = 59
sum t(i) = 9, S(n) = 6n - 4 + 3 sum t(i) = 59

a = [6, 5, 4, 3, 2, 1] (decreasing): S = 77 = (3n^2 + 9n - 8)/2 = 77
sum t(i) = 15 = n(n-1)/2 = 15

Decreasing array (n, ..., 2, 1):
  n | S(n) | (3n^2 + 9n - 8)/2
  1 |    2 | 2
  2 |   11 | 11
  3 |   23 | 23
  4 |   38 | 38
  5 |   56 | 56
  6 |   77 | 77
  7 |  101 | 101
  8 |  128 | 128


**Try it yourself.** Change the array and check the formula $6n - 4 + 3\sum t(i)$.

In [7]:
a = [8, 3, 5, 1, 9, 2, 7]      # <-- change the array
n = len(a)
reset()
print(" i | key | t(i) | array after the iteration")
insertion_sort(a, True)
print_steps(2, 3, 4, 5, 6, 7, 8, 10, 11)
print("6n - 4 + 3 sum t(i) = %d" % (6 * n - 4 + 3 * cnt[7]))

 i | key | t(i) | array after the iteration
 2 |   3 |    1 | [3, 8, 5, 1, 9, 2, 7]
 3 |   5 |    1 | [3, 5, 8, 1, 9, 2, 7]
 4 |   1 |    3 | [1, 3, 5, 8, 9, 2, 7]
 5 |   9 |    0 | [1, 3, 5, 8, 9, 2, 7]
 6 |   2 |    4 | [1, 2, 3, 5, 8, 9, 7]
 7 |   7 |    2 | [1, 2, 3, 5, 7, 8, 9]
Line | Executions
   2 |          1
   3 |          7
   4 |          6
   5 |          6
   6 |         17
   7 |         11
   8 |         11
  10 |          6
  11 |          6
Total steps S = 71
6n - 4 + 3 sum t(i) = 71


## 3. Change of base: ChangeToBase2

**Input:** a natural number $n \ge 1$ in base 10. **Output:** the array $d[0..nd-1]$ of binary digits and the number of digits $nd$. For every $n$: $S(n) = 6\lfloor\log_2 n\rfloor + 11 = \Theta(\log_2 n)$.

Pseudo-code: section 12.4 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/12-running-time/).

In [8]:
def floor_log2(n):                         # floor(log2 n) computed with a loop (it does not count as a step)
    k = 0
    while n > 1:
        n = n // 2
        k = k + 1
    return k


def change_to_base2(n, trace=False):
    nd = floor_log2(n) + 1; cnt[2] += 1
    d = [0] * nd; cnt[3] += 1              # array d[0..nd-1], indexed from 0 as in the pseudo-code
    k = 0; cnt[4] += 1
    cnt[5] += 1                            # the while condition is counted at every evaluation
    while n > 0:
        q = n // 2; cnt[6] += 1
        r = n % 2; cnt[7] += 1
        d[k] = r; cnt[8] += 1
        if trace:                          # trace
            print("%10d | %3d | %3d | %3d | d[%d] = %d" % (k + 1, n, q, r, k, r))
        n = q; cnt[9] += 1
        k = k + 1; cnt[10] += 1
        cnt[5] += 1
    cnt[12] += 1
    return d, nd

In [9]:
# Example of the lecture notes: n = 13
n = 13
reset()
print(" iteration |   n |   q |   r | assignment")
d, nd = change_to_base2(n, True)
print("n = %d = (%s)_2: nd = %d, digits d[nd-1..0] = %s" % (n, "".join(str(x) for x in reversed(d)), nd, "".join(str(x) for x in reversed(d))))
print_steps(2, 3, 4, 5, 6, 7, 8, 9, 10, 12)
print("S(n) = 6 floor(log2 n) + 11 = %d" % (6 * floor_log2(n) + 11))

# The number of steps grows as log2 n
print()
print("       n | nd | S(n) | 6 floor(log2 n) + 11")
for n in [1, 2, 13, 100, 1000, 1000000]:
    reset()
    d, nd = change_to_base2(n)
    print("%8d | %2d | %4d | %d" % (n, nd, sum(cnt), 6 * floor_log2(n) + 11))

 iteration |   n |   q |   r | assignment
         1 |  13 |   6 |   1 | d[0] = 1
         2 |   6 |   3 |   0 | d[1] = 0
         3 |   3 |   1 |   1 | d[2] = 1
         4 |   1 |   0 |   1 | d[3] = 1
n = 13 = (1101)_2: nd = 4, digits d[nd-1..0] = 1101
Line | Executions
   2 |          1
   3 |          1
   4 |          1
   5 |          5
   6 |          4
   7 |          4
   8 |          4
   9 |          4
  10 |          4
  12 |          1
Total steps S = 29
S(n) = 6 floor(log2 n) + 11 = 29

       n | nd | S(n) | 6 floor(log2 n) + 11
       1 |  1 |   11 | 11
       2 |  2 |   17 | 17
      13 |  4 |   29 | 29
     100 |  7 |   47 | 47
    1000 | 10 |   65 | 65
 1000000 | 20 |  125 | 125


**Try it yourself.** Change $n$ and check that $S(n) = 6\lfloor\log_2 n\rfloor + 11$.

In [10]:
n = 2026        # <-- change the number
reset()
d, nd = change_to_base2(n)
print("n = %d = (%s)_2, nd = %d, S = %d" % (n, "".join(str(x) for x in reversed(d)), nd, sum(cnt)))

n = 2026 = (11111101010)_2, nd = 11, S = 71


## 4. Primality test: PrimalityTest

**Input:** a natural number $n$. **Output:** `true` if $n$ is prime, `false` otherwise. In the worst case ($n$ prime) $S(n) = 3\lfloor\sqrt{n}\rfloor + 1 = \Theta(\sqrt{n})$, i.e. $2^{\Theta(|I|)}$ as a function of the input size.

Pseudo-code: section 12.5 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/12-running-time/).

In [11]:
from math import isqrt             # floor(sqrt(n)) in integer arithmetic


def primality_test(n):
    cnt[2] += 1
    if n < 2:
        cnt[3] += 1
        return False
    d = 2; cnt[5] += 1
    cnt[6] += 1                        # the while condition is counted at every evaluation
    while d <= isqrt(n):
        cnt[7] += 1
        if n % d == 0:
            cnt[8] += 1
            return False
        d = d + 1; cnt[10] += 1
        cnt[6] += 1
    cnt[12] += 1
    return True

In [12]:
def risposta(r):
    return "true" if r else "false"


# Example of the lecture notes: a prime and a composite number
n = 97
reset()
r = primality_test(n)
print("n = %d: %s" % (n, risposta(r)))
print_steps(2, 3, 5, 6, 7, 8, 10, 12)
print("S(n) = 3 floor(sqrt(n)) + 1 = %d" % (3 * isqrt(n) + 1))

print()
n = 91
reset()
r = primality_test(n)
p = 2
while n % p != 0:
    p = p + 1
print("n = %d: %s, p = %d" % (n, risposta(r), p))
print_steps(2, 3, 5, 6, 7, 8, 10, 12)
print("S(n) = 3p - 1 = %d, while floor(sqrt(n)) = %d" % (3 * p - 1, isqrt(n)))

print()
n = 1
reset()
r = primality_test(n)
print("n = %d: %s, S = %d" % (n, risposta(r), sum(cnt)))

# The worst case are prime numbers: S grows as sqrt(n), i.e. it doubles for every 2 more bits
print()
print("|I| |        n |  S(n) | 3 floor(sqrt(n)) + 1")
for n in [1021, 4093, 16381, 65521, 262139, 1048573, 4194301, 16777213]:
    reset()
    r = primality_test(n)
    print("%3d | %8d | %5d | %d" % (n.bit_length(), n, sum(cnt), 3 * isqrt(n) + 1))

n = 97: true
Line | Executions
   2 |          1
   3 |          0
   5 |          1
   6 |          9
   7 |          8
   8 |          0
  10 |          8
  12 |          1
Total steps S = 28
S(n) = 3 floor(sqrt(n)) + 1 = 28

n = 91: false, p = 7
Line | Executions
   2 |          1
   3 |          0
   5 |          1
   6 |          6
   7 |          6
   8 |          1
  10 |          5
  12 |          0
Total steps S = 20
S(n) = 3p - 1 = 20, while floor(sqrt(n)) = 9

n = 1: false, S = 2

|I| |        n |  S(n) | 3 floor(sqrt(n)) + 1
 10 |     1021 |    94 | 94
 12 |     4093 |   190 | 190
 14 |    16381 |   382 | 382
 16 |    65521 |   766 | 766
 18 |   262139 |  1534 | 1534
 20 |  1048573 |  3070 | 3070
 22 |  4194301 |  6142 | 6142
 24 | 16777213 | 12286 | 12286


**Try it yourself.** Change $n$: for a prime $S = 3\lfloor\sqrt{n}\rfloor + 1$, for a composite $S = 3p - 1$.

In [13]:
n = 7919        # <-- change the number
reset()
r = primality_test(n)
print("n = %d: %s, S = %d, 3 floor(sqrt(n)) + 1 = %d" % (n, "true" if r else "false", sum(cnt), 3 * isqrt(n) + 1))

n = 7919: true, S = 265, 3 floor(sqrt(n)) + 1 = 265
